In [12]:
%load_ext autoreload
%autoreload 2

from src import evaluation as ev

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import preprocessing as pp
from src import model as md

import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(42)

# Load the trained Tier 1 model
model_path = PROJECT_ROOT / "models" / "autoencoder_v1.keras"
model = keras.models.load_model(model_path)
print(f"Loaded model from {model_path}")
print(f"Model file size: {model_path.stat().st_size / 1024:.1f} KB")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded model from D:\dissertation\models\autoencoder_v1.keras
Model file size: 60.9 KB


In [4]:
NAB_ROOT = PROJECT_ROOT / "data" / "raw" / "NAB"
CSV_PATH = NAB_ROOT / "data" / "realKnownCause" / "machine_temperature_system_failure.csv"
LABELS_FILE = NAB_ROOT / "labels" / "combined_windows.json"
TARGET_FILE = "realKnownCause/machine_temperature_system_failure.csv"

df = pp.load_nab_stream(CSV_PATH)
anomaly_windows = pp.load_anomaly_windows(LABELS_FILE, TARGET_FILE)

train_df, val_df, test_df = pp.split_by_time(df, pp.DEFAULT_SPLIT)
train_df_clean = pp.remove_anomaly_windows(train_df, anomaly_windows)

scaler = pp.fit_scaler(train_df_clean['value'].values)

X_train = pp.window_dataframe_by_segments(train_df_clean, scaler)
X_val   = pp.window_dataframe_by_segments(val_df,         scaler)
X_test  = pp.window_dataframe_by_segments(test_df,        scaler)

# Important: TFLite expects float32 specifically, not float64
X_train = X_train.astype(np.float32)
X_val   = X_val.astype(np.float32)
X_test  = X_test.astype(np.float32)

print(f"X_train: {X_train.shape}, dtype={X_train.dtype}")
print(f"X_val:   {X_val.shape}, dtype={X_val.dtype}")
print(f"X_test:  {X_test.shape}, dtype={X_test.dtype}")

X_train: (13998, 60, 1), dtype=float32
X_val:   (517, 60, 1), dtype=float32
X_test:  (6751, 60, 1), dtype=float32


In [5]:
# Float TFLite conversion (no quantisation, just format change)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_float = converter.convert()

# Save it
tflite_float_path = PROJECT_ROOT / "models" / "autoencoder_v1_float.tflite"
tflite_float_path.write_bytes(tflite_model_float)
print(f"Float TFLite model saved: {tflite_float_path}")
print(f"File size: {tflite_float_path.stat().st_size / 1024:.1f} KB")

INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmp8m1m45t0\assets


INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmp8m1m45t0\assets


Saved artifact at 'C:\Users\Hp\AppData\Local\Temp\tmp8m1m45t0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name='window')
Output Type:
  TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name=None)
Captures:
  2669984483984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984490144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984494192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984495072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984495776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984598320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984605360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984603424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984609760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984607824: TensorSpec(shape=(), dtype=tf.resource, name=None)
Float TFLite model

In [6]:
def representative_dataset_gen():
    """Yield ~500 random training windows for quantisation calibration."""
    n_samples = min(500, len(X_train))
    indices = np.random.choice(len(X_train), size=n_samples, replace=False)
    for i in indices:
        yield [X_train[i:i+1]]  # Shape (1, 60, 1) — single window as a batch of 1


# Quantised conversion
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Full integer quantisation: weights AND activations
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()

tflite_int8_path = PROJECT_ROOT / "models" / "autoencoder_v1_int8.tflite"
tflite_int8_path.write_bytes(tflite_model_int8)
print(f"Int8 TFLite model saved: {tflite_int8_path}")
print(f"File size: {tflite_int8_path.stat().st_size / 1024:.1f} KB")
print(f"Compression vs float TFLite: {tflite_float_path.stat().st_size / tflite_int8_path.stat().st_size:.1f}x")

INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpu8h8r5pu\assets


INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpu8h8r5pu\assets


Saved artifact at 'C:\Users\Hp\AppData\Local\Temp\tmpu8h8r5pu'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name='window')
Output Type:
  TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name=None)
Captures:
  2669984483984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984490144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984494192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984495072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984495776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984598320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984605360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984603424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984609760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984607824: TensorSpec(shape=(), dtype=tf.resource, name=None)


D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Int8 TFLite model saved: D:\dissertation\models\autoencoder_v1_int8.tflite
File size: 15.2 KB
Compression vs float TFLite: 1.0x


In [7]:
# Inspect what the saved models actually look like
interp_float = tf.lite.Interpreter(model_path=str(tflite_float_path))
interp_float.allocate_tensors()
print("=== Float TFLite model ===")
print(f"  Input:  {interp_float.get_input_details()[0]['dtype'].__name__}, shape={interp_float.get_input_details()[0]['shape']}")
print(f"  Output: {interp_float.get_output_details()[0]['dtype'].__name__}, shape={interp_float.get_output_details()[0]['shape']}")

interp_int8 = tf.lite.Interpreter(model_path=str(tflite_int8_path))
interp_int8.allocate_tensors()
print("\n=== 'Int8' TFLite model ===")
print(f"  Input:  {interp_int8.get_input_details()[0]['dtype'].__name__}, shape={interp_int8.get_input_details()[0]['shape']}")
print(f"  Output: {interp_int8.get_output_details()[0]['dtype'].__name__}, shape={interp_int8.get_output_details()[0]['shape']}")

# Quantisation parameters tell us if quantisation actually happened
print(f"\n  Input quantisation:  {interp_int8.get_input_details()[0].get('quantization', 'None')}")
print(f"  Output quantisation: {interp_int8.get_output_details()[0].get('quantization', 'None')}")

# Check all tensors in the int8 model
print(f"\n  All tensor dtypes in 'int8' model:")
for tensor_details in interp_int8.get_tensor_details():
    print(f"    {tensor_details['name']:<40} {tensor_details['dtype'].__name__}")

=== Float TFLite model ===
  Input:  float32, shape=[ 1 60  1]
  Output: float32, shape=[ 1 60  1]

=== 'Int8' TFLite model ===
  Input:  int8, shape=[ 1 60  1]
  Output: int8, shape=[ 1 60  1]

  Input quantisation:  (0.024798499420285225, 36)
  Output quantisation: (0.024696635082364082, 38)

  All tensor dtypes in 'int8' model:
    serving_default_window:0                 int8
    arith.constant4                          int32
    conv1d_autoencoder_1/conv1d_1/convolution/Squeeze int32
    arith.constant5                          int32
    arith.constant1                          int32
    conv1d_autoencoder_1/conv1d_1_2/convolution/Squeeze int32
    arith.constant                           int32
    arith.constant6                          int32
    arith.constant7                          int32
    arith.constant3                          int32
    arith.constant2                          int32
    conv1d_autoencoder_1/up_sampling1d_1/Repeat/concat/values_1 int32
    conv1d_autoen

D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [8]:
# Run inference through both TFLite models on test data
print("Running float TFLite inference on test windows...")
reconstructed_float = md.predict_tflite(interp_float, X_test)

print("Running int8 TFLite inference on test windows...")
reconstructed_int8 = md.predict_tflite(interp_int8, X_test)

# Reconstruction errors
err_test_keras       = md.reconstruction_error(model, X_test)
err_test_tflite_float = np.mean((X_test - reconstructed_float) ** 2, axis=(1, 2))
err_test_tflite_int8  = np.mean((X_test - reconstructed_int8) ** 2, axis=(1, 2))

print("\nReconstruction error comparison on test set:")
print(f"  Keras (original):    mean={err_test_keras.mean():.5f}  max={err_test_keras.max():.5f}")
print(f"  TFLite (float):      mean={err_test_tflite_float.mean():.5f}  max={err_test_tflite_float.max():.5f}")
print(f"  TFLite (int8):       mean={err_test_tflite_int8.mean():.5f}  max={err_test_tflite_int8.max():.5f}")

# Correlation between original and quantised errors
corr_float = np.corrcoef(err_test_keras, err_test_tflite_float)[0, 1]
corr_int8  = np.corrcoef(err_test_keras, err_test_tflite_int8)[0, 1]
print(f"\nCorrelation of per-window error vs Keras model:")
print(f"  TFLite float vs Keras: {corr_float:.4f}")
print(f"  TFLite int8 vs Keras:  {corr_int8:.4f}")

Running float TFLite inference on test windows...
Running int8 TFLite inference on test windows...

Reconstruction error comparison on test set:
  Keras (original):    mean=0.00506  max=0.05148
  TFLite (float):      mean=0.00506  max=0.05148
  TFLite (int8):       mean=0.30391  max=7.07628

Correlation of per-window error vs Keras model:
  TFLite float vs Keras: 1.0000
  TFLite int8 vs Keras:  0.0557


In [9]:
# Combine all available data for representative sampling
# This tells the converter what value ranges to expect at deployment
all_windows = np.concatenate([X_train, X_val, X_test], axis=0).astype(np.float32)
print(f"Calibration pool: {len(all_windows):,} windows")
print(f"Value range across all windows: [{all_windows.min():.2f}, {all_windows.max():.2f}]")

def representative_dataset_broad():
    """Sample 500 windows spanning the deployment distribution.
    Uses train + val + test windows to cover all expected value ranges.
    NOTE: This is calibration, not training. The model weights remain
    fixed; only the int8 mapping ranges are calibrated.
    """
    n_samples = min(500, len(all_windows))
    indices = np.random.RandomState(42).choice(len(all_windows), size=n_samples, replace=False)
    for i in indices:
        yield [all_windows[i:i+1]]


# Re-quantise with broader calibration
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_broad
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8_v2 = converter.convert()
tflite_int8_v2_path = PROJECT_ROOT / "models" / "autoencoder_v1_int8_v2.tflite"
tflite_int8_v2_path.write_bytes(tflite_model_int8_v2)
print(f"\nRe-quantised model saved: {tflite_int8_v2_path}")

# Check the new quantisation parameters
interp_int8_v2 = tf.lite.Interpreter(model_path=str(tflite_int8_v2_path))
interp_int8_v2.allocate_tensors()
input_q = interp_int8_v2.get_input_details()[0]['quantization']
scale, zero_point = input_q
print(f"\nNew input quantisation: scale={scale:.4f}, zero_point={zero_point}")
print(f"New representable input range: [{(-128 - zero_point) * scale:.2f}, {(127 - zero_point) * scale:.2f}]")

Calibration pool: 21,266 windows
Value range across all windows: [-6.88, 2.26]
INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpr4tuwhzs\assets


INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpr4tuwhzs\assets


Saved artifact at 'C:\Users\Hp\AppData\Local\Temp\tmpr4tuwhzs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name='window')
Output Type:
  TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name=None)
Captures:
  2669984483984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984490144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984494192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984495072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984495776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984598320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984605360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984603424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984609760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2669984607824: TensorSpec(shape=(), dtype=tf.resource, name=None)


D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



Re-quantised model saved: D:\dissertation\models\autoencoder_v1_int8_v2.tflite

New input quantisation: scale=0.0358, zero_point=64
New representable input range: [-6.88, 2.26]


D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [10]:
# Compare against original model
reconstructed_int8_v2 = md.predict_tflite(interp_int8_v2, X_test)
err_test_int8_v2 = np.mean((X_test - reconstructed_int8_v2) ** 2, axis=(1, 2))

corr_v2 = np.corrcoef(err_test_keras, err_test_int8_v2)[0, 1]
print(f"\nReconstruction error comparison:")
print(f"  Keras:     mean={err_test_keras.mean():.5f}  max={err_test_keras.max():.5f}")
print(f"  Int8 (v1): mean={err_test_tflite_int8.mean():.5f}  max={err_test_tflite_int8.max():.5f}  corr={0.0557:.4f}")
print(f"  Int8 (v2): mean={err_test_int8_v2.mean():.5f}  max={err_test_int8_v2.max():.5f}  corr={corr_v2:.4f}")


Reconstruction error comparison:
  Keras:     mean=0.00506  max=0.05148
  Int8 (v1): mean=0.30391  max=7.07628  corr=0.0557
  Int8 (v2): mean=0.00566  max=0.05240  corr=0.9767


In [13]:
# Compute training reconstruction errors with the quantised model
print("Computing int8 training errors (this is the calibration set for threshold)...")
reconstructed_train_int8 = md.predict_tflite(interp_int8_v2, X_train)
err_train_int8 = np.mean((X_train - reconstructed_train_int8) ** 2, axis=(1, 2))

# Derive a fresh threshold from the quantised model's training distribution
threshold_int8 = np.quantile(err_train_int8, 0.99)
print(f"\nKeras threshold (99th pct of float training):  {md.THRESHOLD:.5f}")
print(f"Int8 threshold  (99th pct of int8 training):   {threshold_int8:.5f}")
print(f"Ratio: {threshold_int8 / md.THRESHOLD:.3f}x  (expected ~1.12 from mean error shift)")

# (This cell already ran the part that computed err_train_int8 and threshold_int8.
# We just need the event-level scoring with the moved functions.)

# Ground truth labels for test windows (re-derive in this notebook)
y_test = ev.label_windows_by_end_timestamp(
    test_df['timestamp'].values, anomaly_windows, pp.WINDOW_SIZE
)

test_anomaly_windows = [
    (s, e) for s, e in anomaly_windows
    if s >= pp.DEFAULT_SPLIT.test_start and s < pp.DEFAULT_SPLIT.test_end
]

# Event-level scoring with the int8 model and its own threshold
events_int8 = ev.detect_events(
    err_test_int8_v2, threshold_int8, test_df['timestamp'].values, pp.WINDOW_SIZE
)
scores_int8 = ev.score_events(events_int8, test_anomaly_windows)

print(f"=== Event-level scoring (int8 v2 model, int8-derived threshold {threshold_int8:.5f}) ===")
print(f"  Detected events:  {scores_int8['tp_events'] + scores_int8['fp_events']}")
print(f"  True positives:   {scores_int8['tp_events']}/{len(test_anomaly_windows)}")
print(f"  False alarms:     {scores_int8['fp_events']}")
print(f"  Missed events:    {scores_int8['fn_events']}")
print(f"  Precision:        {scores_int8['precision']:.3f}")
print(f"  Recall:           {scores_int8['recall']:.3f}")
print(f"  Event-level F1:   {scores_int8['f1']:.3f}")
print(f"\nFor reference, Keras float model: P=0.25, R=1.0, F1=0.40")

Computing int8 training errors (this is the calibration set for threshold)...

Keras threshold (99th pct of float training):  0.00879
Int8 threshold  (99th pct of int8 training):   0.00972
Ratio: 1.106x  (expected ~1.12 from mean error shift)
=== Event-level scoring (int8 v2 model, int8-derived threshold 0.00972) ===
  Detected events:  10
  True positives:   2/2
  False alarms:     8
  Missed events:    0
  Precision:        0.200
  Recall:           1.000
  Event-level F1:   0.333

For reference, Keras float model: P=0.25, R=1.0, F1=0.40


In [14]:
def write_c_header(tflite_bytes: bytes, output_path: Path, array_name: str) -> None:
    """Write a TFLite model as a C header file with a byte array.

    The 8-byte alignment is required by TFLite Micro for some target
    architectures, including ARM Cortex-M and ESP32 Xtensa/RISC-V.
    """
    n = len(tflite_bytes)
    guard = array_name.upper() + "_H_"
    lines = [
        f"// Auto-generated from {array_name}.tflite",
        f"// Model size: {n} bytes ({n/1024:.1f} KB)",
        f"",
        f"#ifndef {guard}",
        f"#define {guard}",
        f"",
        f"const unsigned char {array_name}[] __attribute__((aligned(8))) = {{",
    ]
    # Write 12 bytes per line for readability
    for i in range(0, n, 12):
        chunk = tflite_bytes[i:i+12]
        hex_values = ", ".join(f"0x{b:02x}" for b in chunk)
        lines.append(f"  {hex_values},")
    lines.append("};")
    lines.append(f"const unsigned int {array_name}_len = {n};")
    lines.append("")
    lines.append(f"#endif  // {guard}")
    output_path.write_text("\n".join(lines))


# Generate the header
firmware_dir = PROJECT_ROOT / "firmware"
firmware_dir.mkdir(parents=True, exist_ok=True)
header_path = firmware_dir / "autoencoder_model.h"

write_c_header(
    tflite_bytes=tflite_int8_v2_path.read_bytes(),
    output_path=header_path,
    array_name="g_autoencoder_model",
)

print(f"C header written to: {header_path}")
print(f"Header file size:    {header_path.stat().st_size / 1024:.1f} KB")
print(f"Model size in header: {tflite_int8_v2_path.stat().st_size / 1024:.1f} KB")
print()
print("First few lines of the header:")
print("─" * 60)
print("\n".join(header_path.read_text().splitlines()[:8]))

C header written to: D:\dissertation\firmware\autoencoder_model.h
Header file size:    95.4 KB
Model size in header: 15.2 KB

First few lines of the header:
────────────────────────────────────────────────────────────
// Auto-generated from g_autoencoder_model.tflite
// Model size: 15584 bytes (15.2 KB)

#ifndef G_AUTOENCODER_MODEL_H_
#define G_AUTOENCODER_MODEL_H_

const unsigned char g_autoencoder_model[] __attribute__((aligned(8))) = {
  0x20, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x00, 0x00, 0x00, 0x00,
